# Gemma 3 12B fine-tuning on Colab (RTX 6000 Pro 95 GB)

Heavy SFT + Abstention-ORPO run for the Governed Clinical-Analytics Copilot.
Flow (per project policy): **heavy training on this big GPU; final inference on the local RTX 3090.**

1. Code + data come from **Google Drive** -> copied to the **local Colab disk** (fast).
2. `HF_TOKEN` is read from **Colab Secrets** (gated Gemma models).
3. Produces `checkpoints/{sft_gemma,orpo_gemma}/adapter_final` -> copied **back to Drive**.
4. Download `orpo_gemma/adapter_final` (~0.5 GB) and run inference on the 3090.

**Before running:** `bash scripts/prepare_colab_bundle.sh` locally, upload
`colab_bundle.tar.gz` to `MyDrive/ehrcopilot/`, and add a Colab secret `HF_TOKEN`.


## 1. Mount Drive + copy bundle to local disk


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
BUNDLE = '/content/drive/MyDrive/ehrcopilot/colab_bundle.tar.gz'
assert os.path.exists(BUNDLE), f'Upload colab_bundle.tar.gz to {BUNDLE} first'
os.makedirs('/content/ehrcopilot', exist_ok=True)
!tar -xzf "$BUNDLE" -C /content/ehrcopilot
os.chdir('/content/ehrcopilot')
print('cwd:', os.getcwd()); !ls


## 2. Install dependencies


In [ ]:
!pip -q install unsloth unsloth_zoo "trl>=0.9" "transformers>=4.50" peft bitsandbytes \
  accelerate datasets sentencepiece timm scikit-learn rank-bm25 sqlglot faiss-cpu einops


## 3. HF token (Colab secret) + environment + GPU check


In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')          # gated Gemma access
os.environ['PYTHONPATH'] = 'src'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch
p = torch.cuda.get_device_properties(0)
VRAM = p.total_memory/1e9
print(f'GPU: {p.name}  {VRAM:.0f} GB')
# Scale batch to VRAM: big GPU -> big batch (fast); 24GB -> bs1.
BS, GA = (16, 1) if VRAM > 70 else (8, 2) if VRAM > 40 else (1, 16)
print('per_device_batch_size =', BS, ' grad_accum =', GA, ' (effective', BS*GA, ')')


## 4. Prepare data (build DB / SFT data if not in the bundle)


In [ ]:
import os
if not os.path.exists('data/mimic_iv_demo.db'):
    print('Building MIMIC-IV-Demo SQLite (open access)...')
    !wget -q -r -N -np -P /content/mimicdl https://physionet.org/files/mimic-iv-demo/2.2/
    !mkdir -p data/mimic-iv-demo && cp -r /content/mimicdl/physionet.org/files/mimic-iv-demo/2.2/. data/mimic-iv-demo/
    !PYTHONPATH=src python -m ehrcopilot.db.build_sqlite data/mimic-iv-demo data/mimic_iv_demo.db
if not os.path.exists('data/ehrsql/sft_train_v2.jsonl'):
    !PYTHONPATH=src python -m ehrcopilot.finetune.prepare_sft \
      --train data/ehrsql/ehrsql/mimic_iii/train.json \
      --valid data/ehrsql/ehrsql/mimic_iii/valid.json \
      --output data/ehrsql/sft_train_v2.jsonl
print('data ready'); !ls -la data/mimic_iv_demo.db data/ehrsql/sft_train_v2.jsonl


## 5. QLoRA SFT (heavy — 3 epochs, big batch on the 95 GB GPU)


In [ ]:
# BS, GA were set from VRAM in the GPU-check cell above.
!PYTHONPATH=src python -m ehrcopilot.finetune.qlora_sft \
  --base-model unsloth/gemma-3-12b-it \
  --data data/ehrsql/sft_train_v2.jsonl --output checkpoints/sft_gemma \
  --epochs 3 --batch-size {BS} --grad-accum {GA}


## 6. Build Abstention-ORPO preference pairs


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.build_pairs \
  --train data/ehrsql/ehrsql/mimic_iii/train.json \
  --valid data/ehrsql/ehrsql/mimic_iii/valid.json \
  --adapter checkpoints/sft_gemma/adapter_final \
  --output data/ehrsql/gemma_orpo_pairs.jsonl --max-answerable 500 --verify-execution


## 7. Abstention-ORPO (calibrates [ABSTAIN] -> positive RS)


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.abstention_dpo \
  --pairs data/ehrsql/gemma_orpo_pairs.jsonl \
  --adapter checkpoints/sft_gemma/adapter_final --output checkpoints/orpo_gemma \
  --epochs 2 --max-length 1536


## 8. Copy adapters back to Drive (for 3090 inference)


In [ ]:
!mkdir -p /content/drive/MyDrive/ehrcopilot/checkpoints
!cp -r checkpoints/sft_gemma  /content/drive/MyDrive/ehrcopilot/checkpoints/
!cp -r checkpoints/orpo_gemma /content/drive/MyDrive/ehrcopilot/checkpoints/
print('Saved. Download checkpoints/orpo_gemma/adapter_final and run inference on the RTX 3090.')


## 9. (optional) Quick eval on the 75-question subset


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.eval.harness \
  data/ehrsql/ehrsql/mimic_iii/test_cmp75.json \
  --model checkpoints/orpo_gemma/adapter_final \
  --few-shot data/ehrsql/ehrsql/mimic_iii/train.json \
  --retrieval-mode classifier --repair --output tests/evalgen/colab_orpo_ex.json
import json; print(json.load(open('tests/evalgen/colab_orpo_ex.json')))
